# Tulya Experiment 2 v3 — Shared-Regime Zero-Shot Prognosis

v2 is closed as a failed preflight. This is a **new experiment**, not a repair of v2.

v3 uses only the three regimes that were empirically shared across heterogeneous domains:

- `no_event`
- `stagnation`
- `overfit_or_memorization`

The v2 seed-0 development runs are excluded. v3 uses fresh seeds **100–105**.

Total corpus: **60 runs** (10 frozen domain/recipe cells × 6 seeds).

There is no further recipe-tuning round. If the v3 corpus validity gate fails, v3 stops.

Enable **GPU T4 ×2** and **Internet**.

In [1]:
import os, sys, subprocess, importlib, json, time

REPO="/kaggle/working/tulya-training-dynamics"
if os.path.exists(os.path.join(REPO,".git")):
    subprocess.run(["git","-C",REPO,"fetch","origin","main"],check=True)
    subprocess.run(["git","-C",REPO,"reset","--hard","origin/main"],check=True)
else:
    subprocess.run(["git","clone","https://github.com/Vedsaga/tulya-training-dynamics.git",REPO],check=True)

os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0,REPO)

for m in ["experiment2_eval","experiment2_v3","experiment2_core","kaggle_grokking_experiment"]:
    sys.modules.pop(m,None)
importlib.invalidate_caches()

import torch, pandas as pd
commit=subprocess.check_output(["git","-C",REPO,"rev-parse","HEAD"],text=True).strip()
print("commit:",commit)
print("CUDA:",torch.cuda.is_available())
print("visible GPUs:",torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}:",torch.cuda.get_device_name(i))
if torch.cuda.device_count()<2:
    print("WARNING: v3 still runs, but intended speedup requires 2 GPUs.")

From https://github.com/Vedsaga/tulya-training-dynamics
 * branch            main       -> FETCH_HEAD
   27ae248..ac8cb1a  main       -> origin/main


HEAD is now at ac8cb1a Compile v3 experiment module in CI
commit: ac8cb1a2a16c68c086493868966ac2e67f1554bf
CUDA: True
visible GPUs: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


## 1. Frozen v3 preregistration

In [2]:
print(open("EXPERIMENT2_V3_PREREGISTRATION.md").read())

# Experiment 2 v3 — Shared-Regime Zero-Shot Prognosis

Created after Experiment 2 v2 failed its repaired preflight. No v2 A/B/C/D predictor evaluation was performed.

## What v2 falsified

The stronger assumption that one fine-grained event ontology containing divergence can be naturally instantiated across all cheap domains was not supported. Divergence remained modular-Transformer-only after the single allowed v2 configuration repair.

v3 does not rewrite v2. It asks a narrower fresh question.

## Claim under test

Task-agnostic normalized training telemetry can forecast **shared future training regimes that empirically exist across heterogeneous domains**, zero-shot on an unseen task/architecture, and outperform strong raw telemetry.

The shared outcomes are frozen as:

- `no_event` — right-censored healthy completion
- `stagnation`
- `overfit_or_memorization`

No `divergence` or `delayed_improvement` is part of the v3 target ontology.

## Why these three

The v2 repaired seed-0 dev

## 2. Run the fresh 60-run corpus

This is the scientific corpus.

There is **no seed-0 preflight** in v3, because seed 0 was already used to design the recipe set. We run fresh untouched seeds 100–105 directly.

The runner is resumable. Completed v3 runs are reused if Kaggle disconnects.

In [3]:
import importlib
import experiment2_v3
experiment2_v3=importlib.reload(experiment2_v3)

ROOT="/kaggle/working/tulya_exp2_v3"
start=time.time()
manifest=experiment2_v3.run_v3_suite(ROOT)
print("v3 suite wall time:",time.time()-start)

display(manifest.groupby(
    ["domain","recipe_name","recipe_expected_event","event","event_observed"]
).size().rename("runs").reset_index())

Launching 60 v3 runs across 2 dynamically balanced GPU workers: [0, 1]
[GPU 0] start modular_transformer__healthy__seed100
[GPU 1] start modular_transformer__healthy__seed101
[GPU 0] done modular_transformer__healthy__seed100: no_event observed=False @ 1.000, 1012.1s
[GPU 0] start modular_transformer__healthy__seed102
[GPU 1] done modular_transformer__healthy__seed101: no_event observed=False @ 1.000, 1014.9s
[GPU 1] start modular_transformer__healthy__seed103
[GPU 0] done modular_transformer__healthy__seed102: no_event observed=False @ 1.000, 1012.5s
[GPU 0] start modular_transformer__healthy__seed104
[GPU 1] done modular_transformer__healthy__seed103: no_event observed=False @ 1.000, 1016.2s
[GPU 1] start modular_transformer__healthy__seed105
[GPU 0] done modular_transformer__healthy__seed104: no_event observed=False @ 1.000, 1012.8s
[GPU 0] start fashion_mnist_mlp__healthy__seed100
[GPU 1] done modular_transformer__healthy__seed105: no_event observed=False @ 1.000, 1016.1s
[GPU 1] s

,domain,recipe_name,recipe_expected_event,event,event_observed,runs
0,cifar10_cnn,no_event,no_event,no_event,False,6
1,cifar10_cnn,overfit_or_memorization,overfit_or_memorization,overfit_or_memorization,True,6
2,cifar10_cnn,stagnation,stagnation,stagnation,True,6
3,fashion_mnist_mlp,no_event,no_event,no_event,False,6
4,fashion_mnist_mlp,stagnation,stagnation,stagnation,True,6
5,modular_transformer,no_event,no_event,no_event,False,6
6,modular_transformer,overfit_or_memorization,overfit_or_memorization,overfit_or_memorization,True,6
7,synthetic_sequence_gru,no_event,no_event,no_event,False,6
8,synthetic_sequence_gru,overfit_or_memorization,overfit_or_memorization,overfit_or_memorization,True,6
9,synthetic_sequence_gru,stagnation,stagnation,stagnation,True,6


## 3. Corpus validity gate

The full 60-run corpus must satisfy the frozen v3 validity rules before any A/B/C/D evaluation.

If this fails, stop. There is no recipe repair inside v3.

In [4]:
problems,recipe_stability=experiment2_v3.validate_v3_manifest(manifest)

print("Recipe stability:")
display(recipe_stability)

print("\nObserved outcomes by domain:")
display(manifest.groupby(["domain","event"]).size().rename("runs").reset_index())

if problems:
    print("\nV3 CORPUS: INVALID_DATA_GENERATION")
    for p in problems:
        print(" -",p)
    v3_valid=False
else:
    print("\nV3 CORPUS: VALID — blind evaluation allowed.")
    v3_valid=True

Recipe stability:


,domain,recipe_name,recipe_expected_event,matches,runs
0,cifar10_cnn,no_event,no_event,6,6
1,cifar10_cnn,overfit_or_memorization,overfit_or_memorization,6,6
2,cifar10_cnn,stagnation,stagnation,6,6
3,fashion_mnist_mlp,no_event,no_event,6,6
4,fashion_mnist_mlp,stagnation,stagnation,6,6
5,modular_transformer,no_event,no_event,6,6
6,modular_transformer,overfit_or_memorization,overfit_or_memorization,6,6
7,synthetic_sequence_gru,no_event,no_event,6,6
8,synthetic_sequence_gru,overfit_or_memorization,overfit_or_memorization,6,6
9,synthetic_sequence_gru,stagnation,stagnation,6,6



Observed outcomes by domain:


,domain,event,runs
0,cifar10_cnn,no_event,6
1,cifar10_cnn,overfit_or_memorization,6
2,cifar10_cnn,stagnation,6
3,fashion_mnist_mlp,no_event,6
4,fashion_mnist_mlp,stagnation,6
5,modular_transformer,no_event,6
6,modular_transformer,overfit_or_memorization,6
7,synthetic_sequence_gru,no_event,6
8,synthetic_sequence_gru,overfit_or_memorization,6
9,synthetic_sequence_gru,stagnation,6



V3 CORPUS: VALID — blind evaluation allowed.


## 4. Blind zero-shot evaluation

Core verdict still uses only A/B/C:

- A — learning curves
- B — raw telemetry
- C — normalized/canonical telemetry

D = raw + normalized is diagnostic only and cannot rescue C.

The evaluator also reports bootstrap confidence intervals, within-domain grouped CV, domain identifiability, and false-stop compute utility.

In [5]:
if not v3_valid:
    raise RuntimeError("v3 corpus invalid; blind evaluator intentionally blocked.")

import experiment2_eval
experiment2_eval=importlib.reload(experiment2_eval)

forecast_table=experiment2_eval.build_table(ROOT)
folds,result,within_domain,domain_identity=experiment2_eval.evaluate(ROOT)

print("\nZERO-SHOT LEAVE-ONE-DOMAIN-OUT:")
display(folds)

print("\nWITHIN-DOMAIN GROUPED DIAGNOSTIC:")
display(within_domain)

print("\nDOMAIN-IDENTITY DIAGNOSTIC:")
display(domain_identity)

print("\nDIAGNOSTIC INTERPRETATION:")
print(json.dumps(result.get("diagnostics",{}),indent=2))

print("\nCORE RESULT:")
print(json.dumps(result,indent=2))


ZERO-SHOT LEAVE-ONE-DOMAIN-OUT:


,held_out_domain,system,macro_auroc,macro_auroc_ci_low,macro_auroc_ci_high,brier,ece,event_time_mae,warning_threshold,warning_fpr,median_warning_lead,n_test_prefixes,n_test_runs,classes_test,economic_tp_saved_fraction_sum,economic_fp_remaining_fraction_sum,economic_net_fraction_m1,economic_net_fraction_m1_5,economic_net_fraction_m2,economic_break_even_false_stop_multiplier
0,modular_transformer,A_learning_curve,0.500000,0.500000,0.500000,1.000000,0.500000,NaN,0.195472,0.0,NaN,48,12,"no_event,overfit_or_memorization",0.000000,0.0,0.000000,0.000000,0.000000,NaN
1,modular_transformer,B_raw_telemetry,0.500000,0.500000,0.500000,1.000000,0.500000,0.833333,0.078395,1.0,0.316667,48,12,"no_event,overfit_or_memorization",1.900000,5.4,-0.291667,-0.516667,-0.741667,0.351852
2,modular_transformer,C_canonical,0.723958,0.578211,0.842373,0.988638,0.495310,0.058557,0.051459,1.0,0.316667,48,12,"no_event,overfit_or_memorization",1.900000,5.4,-0.291667,-0.516667,-0.741667,0.351852
3,modular_transformer,D_hybrid_diagnostic,0.500000,0.500000,0.500000,1.000000,0.500000,0.833333,0.034632,1.0,0.316667,48,12,"no_event,overfit_or_memorization",1.900000,5.4,-0.291667,-0.516667,-0.741667,0.351852
4,fashion_mnist_mlp,A_learning_curve,1.000000,1.000000,1.000000,0.014382,0.041919,0.039226,0.567208,0.0,0.511111,48,12,"no_event,stagnation",3.066667,0.0,0.255556,0.255556,0.255556,NaN
5,fashion_mnist_mlp,B_raw_telemetry,1.000000,1.000000,1.000000,0.169112,0.055489,0.030732,0.145526,1.0,0.511111,48,12,"no_event,stagnation",3.066667,4.8,-0.144444,-0.344444,-0.544444,0.638889
6,fashion_mnist_mlp,C_canonical,1.000000,1.000000,1.000000,0.120278,0.070724,0.041790,0.073464,1.0,0.511111,48,12,"no_event,stagnation",3.066667,5.4,-0.194444,-0.419444,-0.644444,0.567901
7,fashion_mnist_mlp,D_hybrid_diagnostic,1.000000,1.000000,1.000000,0.041663,0.074200,0.045792,0.036709,1.0,0.511111,48,12,"no_event,stagnation",3.066667,5.4,-0.194444,-0.419444,-0.644444,0.567901
8,cifar10_cnn,A_learning_curve,0.949074,0.920664,0.976848,0.261756,0.078428,0.027156,0.498624,1.0,0.429720,72,18,"no_event,overfit_or_memorization,stagnation",5.156643,5.1,0.003147,-0.138520,-0.280186,1.011107
9,cifar10_cnn,B_raw_telemetry,0.998264,0.993666,1.000000,0.412865,0.207111,0.038444,0.115979,0.0,0.429720,72,18,"no_event,overfit_or_memorization,stagnation",5.156643,0.0,0.286480,0.286480,0.286480,NaN



WITHIN-DOMAIN GROUPED DIAGNOSTIC:


,domain,system,in_domain_grouped_macro_auroc,n_splits,n_runs
0,modular_transformer,A_learning_curve,1.0,4,12
1,modular_transformer,B_raw_telemetry,1.0,4,12
2,modular_transformer,C_canonical,1.0,4,12
3,modular_transformer,D_hybrid_diagnostic,1.0,4,12
4,fashion_mnist_mlp,A_learning_curve,1.0,4,12
5,fashion_mnist_mlp,B_raw_telemetry,1.0,4,12
6,fashion_mnist_mlp,C_canonical,1.0,4,12
7,fashion_mnist_mlp,D_hybrid_diagnostic,1.0,4,12
8,cifar10_cnn,A_learning_curve,1.0,4,18
9,cifar10_cnn,B_raw_telemetry,1.0,4,18



DOMAIN-IDENTITY DIAGNOSTIC:


,system,domain_identity_macro_auroc,n_runs
0,B_raw_telemetry,0.999809,60
1,C_canonical,1.000000,60
2,D_hybrid_diagnostic,1.000000,60



DIAGNOSTIC INTERPRETATION:
{
  "diagnostic_pattern": "LITTLE_INCREMENTAL_TELEMETRY_SIGNAL",
  "mean_zero_shot_auc": {
    "A_learning_curve": 0.8622685185185185,
    "B_raw_telemetry": 0.871021412037037,
    "C_canonical": 0.8600260416666666,
    "D_hybrid_diagnostic": 0.8749276620370371
  },
  "mean_in_domain_auc": {
    "A_learning_curve": 1.0,
    "B_raw_telemetry": 1.0,
    "C_canonical": 1.0,
    "D_hybrid_diagnostic": 1.0
  },
  "in_domain_minus_zero_shot_gap": {
    "A_learning_curve": 0.1377314814814815,
    "B_raw_telemetry": 0.12897858796296302,
    "C_canonical": 0.13997395833333337,
    "D_hybrid_diagnostic": 0.1250723379629629
  },
  "domain_identity_auc": {
    "B_raw_telemetry": 0.9998088210978836,
    "C_canonical": 1.0,
    "D_hybrid_diagnostic": 1.0
  }
}

CORE RESULT:
{
  "verdict": "KILL_PRODUCT_DIRECTION",
  "metrics": {
    "mean_auc_canonical": 0.8600260416666666,
    "min_domain_auc_canonical": 0.7161458333333334,
    "mean_auc_delta_vs_raw": -0.010995370370370

## 5. Decision

Interpret the evaluator's core result using the v3 preregistration:

- core C-vs-B gates pass → **CONTINUE_TO_REAL_LM_EXPERIMENT**
- any core gate fails → **KILL_CANONICAL_TRAINING_DYNAMICS_THESIS**

Do not add new features, learned canonicalization, new thresholds, or new event classes after seeing this result.

In [6]:
if result["verdict"]=="CONTINUE_TO_EXPERIMENT_3":
    v3_verdict="CONTINUE_TO_REAL_LM_EXPERIMENT"
else:
    v3_verdict="KILL_CANONICAL_TRAINING_DYNAMICS_THESIS"

print("V3 VERDICT:",v3_verdict)
print("\nArtifacts:")
for name in [
    "manifest.csv",
    "forecast_table.csv",
    "evaluation_folds.csv",
    "evaluation_summary.json",
    "diagnostic_in_domain.csv",
    "diagnostic_domain_identity.csv",
    "diagnostic_interpretation.json",
]:
    print(os.path.join(ROOT,name))

V3 VERDICT: KILL_CANONICAL_TRAINING_DYNAMICS_THESIS

Artifacts:
/kaggle/working/tulya_exp2_v3/manifest.csv
/kaggle/working/tulya_exp2_v3/forecast_table.csv
/kaggle/working/tulya_exp2_v3/evaluation_folds.csv
/kaggle/working/tulya_exp2_v3/evaluation_summary.json
/kaggle/working/tulya_exp2_v3/diagnostic_in_domain.csv
/kaggle/working/tulya_exp2_v3/diagnostic_domain_identity.csv
/kaggle/working/tulya_exp2_v3/diagnostic_interpretation.json
